In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Find the PaySim CSV whether Jupyter runs from the project
# root or from inside the notebooks directory.
csv_files = list(Path("data/raw").glob("*.csv"))
csv_files += list(Path("../data/raw").glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No CSV found in data/raw. Check the PaySim dataset location."
    )

DATA_PATH = csv_files[0]
print("Dataset:", DATA_PATH)

# Load only fields available for pre-transaction scoring.
raw = pd.read_csv(
    DATA_PATH,
    usecols=[
        "step", "type", "amount",
        "oldbalanceOrg", "oldbalanceDest", "isFraud"
    ],
    dtype={
        "step": "int16",
        "type": "category",
        "amount": "float32",
        "oldbalanceOrg": "float32",
        "oldbalanceDest": "float32",
        "isFraud": "uint8"
    }
)

# Feature engineering
features = pd.DataFrame(index=raw.index)

features["amount"] = raw["amount"]
features["amount_log"] = np.log1p(
    raw["amount"].astype("float64")
).astype("float32")

features["hour"] = (raw["step"] % 24).astype("int8")
features["day"] = (raw["step"] // 24).astype("int16")

# Pre-transaction balances (log scale helps handle large values).
features["origin_balance_log"] = np.log1p(
    raw["oldbalanceOrg"].astype("float64")
).astype("float32")

features["destination_balance_log"] = np.log1p(
    raw["oldbalanceDest"].astype("float64")
).astype("float32")

# Ratios: how large is this transaction relative to the
# available pre-transaction balance?
features["origin_amount_ratio"] = (
    raw["amount"] / (raw["oldbalanceOrg"] + 1.0)
).clip(0, 10).astype("float32")

features["destination_amount_ratio"] = (
    raw["amount"] / (raw["oldbalanceDest"] + 1.0)
).clip(0, 10).astype("float32")

features["origin_balance_zero"] = (
    raw["oldbalanceOrg"] == 0
).astype("uint8")

features["destination_balance_zero"] = (
    raw["oldbalanceDest"] == 0
).astype("uint8")

# Encode transaction type.
type_features = pd.get_dummies(
    raw["type"],
    prefix="type",
    dtype="uint8"
)

features = pd.concat([features, type_features], axis=1)
target = raw["isFraud"].astype("uint8")

# Preserve the split strategy established earlier:
# development: step <= 520
# temporal validation: steps 521–631
# final temporal test: steps 632–743
development_positions = np.flatnonzero(
    (raw["step"] <= 520).to_numpy()
)

train_positions, validation_positions = train_test_split(
    development_positions,
    test_size=0.20,
    stratify=target.iloc[development_positions],
    random_state=42
)

X_train_full = features.iloc[train_positions]
y_train_full = target.iloc[train_positions]

# Limit training size to speed up model experimentation.
sample_size = min(1_000_000, len(X_train_full))
X_train = X_train_full.sample(
    n=sample_size,
    random_state=42
)
y_train = target.loc[X_train.index]

# Internal validation: model selection and threshold exploration.
X_val = features.iloc[validation_positions]
y_val = target.iloc[validation_positions]

# Later-period validation: measure distribution-shift impact.
time_val_positions = np.flatnonzero(
    raw["step"].between(521, 631).to_numpy()
)
X_time_val = features.iloc[time_val_positions]
y_time_val = target.iloc[time_val_positions]

# Do not evaluate this final test set during model selection.
test_positions = np.flatnonzero(
    raw["step"].between(632, 743).to_numpy()
)
X_final_test = features.iloc[test_positions]
y_final_test = target.iloc[test_positions]

print("\nFeature count:", features.shape[1])
print("Training rows:", len(X_train))
print("Training fraud cases:", int(y_train.sum()))
print("Internal validation rows:", len(X_val))
print("Temporal validation rows:", len(X_time_val))
print("Final test rows reserved:", len(X_final_test))
print("\nTraining fraud rate:", round(y_train.mean() * 100, 4), "%")
print("Validation fraud rate:", round(y_val.mean() * 100, 4), "%")
print("Temporal validation fraud rate:",
      round(y_time_val.mean() * 100, 4), "%")

Dataset: ..\data\raw\PS_20174392719_1491204439457_log.csv

Feature count: 15
Training rows: 1000000
Training fraud cases: 965
Internal validation rows: 1216402
Temporal validation rows: 191147
Final test rows reserved: 89466

Training fraud rate: 0.0965 %
Validation fraud rate: 0.095 %
Temporal validation fraud rate: 0.6173 %


In [2]:
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# --------------------------------------------------
# 1. Retain ALL fraud cases from the training pool
# --------------------------------------------------
fraud_indices = y_train_full.index[y_train_full == 1]
legitimate_indices = y_train_full.index[y_train_full == 0]

n_fraud = len(fraud_indices)
n_legitimate_to_keep = min(
    len(legitimate_indices),
    100 * n_fraud
)

# Sample legitimate rows by index to reduce memory use.
legitimate_sample_indices = (
    pd.Series(legitimate_indices)
    .sample(n=n_legitimate_to_keep, random_state=42)
    .to_numpy()
)

train_indices = np.concatenate([
    fraud_indices.to_numpy(),
    legitimate_sample_indices
])

np.random.default_rng(42).shuffle(train_indices)

X_train = X_train_full.loc[train_indices]
y_train = y_train_full.loc[train_indices]

print("TRAINING DATA")
print("Rows:", f"{len(X_train):,}")
print("Fraud cases:", f"{int(y_train.sum()):,}")
print("Legitimate cases:", f"{int((y_train == 0).sum()):,}")
print("Fraud rate:", f"{y_train.mean() * 100:.3f}%")

# --------------------------------------------------
# 2. Train a non-linear model
# --------------------------------------------------
fraud_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

fraud_model.fit(X_train, y_train)

# --------------------------------------------------
# 3. Evaluate on the ORIGINAL internal validation set
#    Do not resample or rebalance this set.
# --------------------------------------------------
validation_prob = fraud_model.predict_proba(X_val)[:, 1]

print("\nINTERNAL VALIDATION")
print("ROC-AUC:", round(
    roc_auc_score(y_val, validation_prob), 4
))
print("Average precision (PR-AUC metric):", round(
    average_precision_score(y_val, validation_prob), 4
))

# Select a threshold from actual validation predictions,
# rather than relying on an arbitrary fixed threshold grid.
precisions, recalls, thresholds = precision_recall_curve(
    y_val, validation_prob
)

f1_values = (
    2 * precisions[:-1] * recalls[:-1]
    / (precisions[:-1] + recalls[:-1] + 1e-12)
)

best_idx = int(np.argmax(f1_values))
best_threshold = float(thresholds[best_idx])

validation_pred = (
    validation_prob >= best_threshold
).astype("uint8")

print("\nBEST INTERNAL-VALIDATION F1 THRESHOLD:", best_threshold)
print("Precision:", round(
    precision_score(y_val, validation_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_val, validation_pred, zero_division=0), 4
))
print("F1:", round(
    f1_score(y_val, validation_pred, zero_division=0), 4
))
print("Confusion matrix:")
print(confusion_matrix(y_val, validation_pred))

# --------------------------------------------------
# 4. Evaluate on the LATER validation period.
#    We still do NOT touch the final test set.
# --------------------------------------------------
time_val_prob = fraud_model.predict_proba(X_time_val)[:, 1]
time_val_pred = (
    time_val_prob >= best_threshold
).astype("uint8")

print("\nLATER-PERIOD VALIDATION")
print("ROC-AUC:", round(
    roc_auc_score(y_time_val, time_val_prob), 4
))
print("Average precision (PR-AUC metric):", round(
    average_precision_score(y_time_val, time_val_prob), 4
))
print("Precision at selected threshold:", round(
    precision_score(y_time_val, time_val_pred, zero_division=0), 4
))
print("Recall at selected threshold:", round(
    recall_score(y_time_val, time_val_pred, zero_division=0), 4
))
print("F1 at selected threshold:", round(
    f1_score(y_time_val, time_val_pred, zero_division=0), 4
))
print("Confusion matrix:")
print(confusion_matrix(y_time_val, time_val_pred))

TRAINING DATA
Rows: 467,125
Fraud cases: 4,625
Legitimate cases: 462,500
Fraud rate: 0.990%

INTERNAL VALIDATION
ROC-AUC: 0.9998
Average precision (PR-AUC metric): 0.9946

BEST INTERNAL-VALIDATION F1 THRESHOLD: 0.9647442597113763
Precision: 0.9956
Recall: 0.9827
F1: 0.9891
Confusion matrix:
[[1215241       5]
 [     20    1136]]

LATER-PERIOD VALIDATION
ROC-AUC: 1.0
Average precision (PR-AUC metric): 0.9999
Precision at selected threshold: 0.9983
Recall at selected threshold: 0.9898
F1 at selected threshold: 0.994
Confusion matrix:
[[189965      2]
 [    12   1168]]


In [3]:

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)

# A: Core transaction features, excluding all balance information.
base_cols = [
    c for c in X_train.columns
    if c in ["amount", "amount_log", "hour", "day"]
    or c.startswith("type_")
]

# B: Pre-transaction balances and zero-balance indicators.
balance_cols = [
    "origin_balance_log",
    "destination_balance_log",
    "origin_balance_zero",
    "destination_balance_zero",
]

# C: The ratio features that may expose a simulator-specific shortcut.
ratio_cols = [
    "origin_amount_ratio",
    "destination_amount_ratio",
]

feature_sets = {
    "A_Core_only": base_cols,
    "B_Core_plus_balances": base_cols + balance_cols,
    "C_All_features": base_cols + balance_cols + ratio_cols,
}

results = []

for name, cols in feature_sets.items():

    if name == "C_All_features":
        # Reuse the full-feature model trained in the previous cell.
        model = fraud_model
        Xv = X_val
        Xt = X_time_val
    else:
        model = HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        )

        model.fit(X_train[cols], y_train)
        Xv = X_val[cols]
        Xt = X_time_val[cols]

    # Internal validation: select threshold here.
    val_prob = model.predict_proba(Xv)[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_val, val_prob
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )

    best_idx = int(f1_values.argmax())
    threshold = float(thresholds[best_idx])
    val_pred = (val_prob >= threshold).astype("uint8")

    # Later-period validation: evaluate the selected threshold.
    time_prob = model.predict_proba(Xt)[:, 1]
    time_pred = (time_prob >= threshold).astype("uint8")

    results.append({
        "experiment": name,
        "features": len(cols),
        "threshold": threshold,
        "validation_AP": average_precision_score(y_val, val_prob),
        "validation_ROC_AUC": roc_auc_score(y_val, val_prob),
        "validation_precision": precision_score(
            y_val, val_pred, zero_division=0
        ),
        "validation_recall": recall_score(
            y_val, val_pred, zero_division=0
        ),
        "temporal_AP": average_precision_score(
            y_time_val, time_prob
        ),
        "temporal_ROC_AUC": roc_auc_score(
            y_time_val, time_prob
        ),
        "temporal_precision": precision_score(
            y_time_val, time_pred, zero_division=0
        ),
        "temporal_recall": recall_score(
            y_time_val, time_pred, zero_division=0
        ),
    })

ablation_results = pd.DataFrame(results)

display(
    ablation_results.round(4).sort_values(
        "temporal_AP", ascending=False
    )
)


,experiment,features,threshold,validation_AP,validation_ROC_AUC,validation_precision,validation_recall,temporal_AP,temporal_ROC_AUC,temporal_precision,temporal_recall
2,C_All_features,15,0.9647,0.9946,0.9998,0.9956,0.9827,0.9999,1.0000,0.9983,0.9898
1,B_Core_plus_balances,13,0.8053,0.8604,0.9989,0.8486,0.7760,0.9148,0.9988,0.8425,0.8568
0,A_Core_only,9,0.7657,0.5170,0.9646,0.7574,0.4455,0.3486,0.9326,0.1740,0.4907


In [4]:

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)

ratio_experiments = {
    "B_plus_origin_ratio": (
        base_cols + balance_cols + ["origin_amount_ratio"]
    ),
    "B_plus_destination_ratio": (
        base_cols + balance_cols + ["destination_amount_ratio"]
    ),
}

ratio_results = []

for name, cols in ratio_experiments.items():
    model = HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    )

    model.fit(X_train[cols], y_train)

    # Select threshold only on internal validation.
    val_prob = model.predict_proba(X_val[cols])[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_val, val_prob
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )

    best_idx = int(f1_values.argmax())
    threshold = float(thresholds[best_idx])

    # Evaluate the same threshold on the later period.
    time_prob = model.predict_proba(X_time_val[cols])[:, 1]
    time_pred = (time_prob >= threshold).astype("uint8")

    ratio_results.append({
        "experiment": name,
        "validation_AP": average_precision_score(
            y_val, val_prob
        ),
        "temporal_AP": average_precision_score(
            y_time_val, time_prob
        ),
        "temporal_ROC_AUC": roc_auc_score(
            y_time_val, time_prob
        ),
        "temporal_precision": precision_score(
            y_time_val, time_pred, zero_division=0
        ),
        "temporal_recall": recall_score(
            y_time_val, time_pred, zero_division=0
        ),
    })

display(pd.DataFrame(ratio_results).round(4))


,experiment,validation_AP,temporal_AP,temporal_ROC_AUC,temporal_precision,temporal_recall
0,B_plus_origin_ratio,0.9948,0.9999,1.0000,0.9983,0.9898
1,B_plus_destination_ratio,0.8615,0.9160,0.9989,0.8239,0.8644


In [5]:

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Primary model: exclude both engineered ratio features.
selected_features = base_cols + balance_cols

final_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42,
)

# Train using all fraud cases and the sampled legitimate cases.
final_model.fit(X_train[selected_features], y_train)

# Select the decision threshold ONLY on internal validation.
val_prob = final_model.predict_proba(
    X_val[selected_features]
)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(
    y_val, val_prob
)

f1_values = (
    2 * precisions[:-1] * recalls[:-1]
    / (precisions[:-1] + recalls[:-1] + 1e-12)
)

best_idx = int(f1_values.argmax())
selected_threshold = float(thresholds[best_idx])

print("Selected threshold:", round(selected_threshold, 6))
print("Validation AP:", round(
    average_precision_score(y_val, val_prob), 4
))

# FINAL TEST: evaluate once, without changing the threshold.
test_prob = final_model.predict_proba(
    X_final_test[selected_features]
)[:, 1]

test_pred = (test_prob >= selected_threshold).astype("uint8")

print("\nFINAL TEMPORAL TEST RESULTS")
print("ROC-AUC:", round(
    roc_auc_score(y_final_test, test_prob), 4
))
print("Average precision:", round(
    average_precision_score(y_final_test, test_prob), 4
))
print("Precision:", round(
    precision_score(y_final_test, test_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_final_test, test_pred, zero_division=0), 4
))
print("F1:", round(
    f1_score(y_final_test, test_pred, zero_division=0), 4
))

print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_final_test, test_pred))

print("\nClassification report:")
print(classification_report(
    y_final_test, test_pred,
    target_names=["Legitimate", "Fraud"],
    zero_division=0,
))


Selected threshold: 0.805306
Validation AP: 0.8604

FINAL TEMPORAL TEST RESULTS
ROC-AUC: 0.9985
Average precision: 0.9363
Precision: 0.8979
Recall: 0.8355
F1: 0.8655

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[88095   119]
 [  206  1046]]

Classification report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     88214
       Fraud       0.90      0.84      0.87      1252

    accuracy                           1.00     89466
   macro avg       0.95      0.92      0.93     89466
weighted avg       1.00      1.00      1.00     89466



In [6]:

from pathlib import Path
import json
import joblib

# Find the project root whether Jupyter runs from
# the project folder or from the notebooks folder.
cwd = Path.cwd().resolve()

if (cwd / "data" / "raw").exists():
    project_root = cwd
elif (cwd.parent / "data" / "raw").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find the project root containing data/raw."
    )

artifacts_dir = project_root / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = artifacts_dir / "fraud_model.joblib"
metadata_path = artifacts_dir / "model_metadata.json"

# Save the exact selected model and its decision threshold.
model_bundle = {
    "model": final_model,
    "features": list(selected_features),
    "threshold": float(selected_threshold),
    "model_name": "HistGradientBoostingClassifier",
    "feature_version": "v1",
}

joblib.dump(model_bundle, model_path)

metadata = {
    "project": "FraudShield",
    "model_name": "HistGradientBoostingClassifier",
    "features": list(selected_features),
    "threshold": float(selected_threshold),
    "threshold_selection": (
        "F1 maximization on internal validation data"
    ),
    "training_strategy": (
        "All training fraud cases plus 100 sampled "
        "legitimate cases per fraud case"
    ),
    "evaluation_dataset": (
        "PaySim synthetic financial transactions"
    ),
    "final_temporal_test": {
        "roc_auc": 0.9985,
        "average_precision": 0.9363,
        "fraud_precision": 0.8979,
        "fraud_recall": 0.8355,
        "fraud_f1": 0.8655,
        "true_positives": 1046,
        "false_positives": 119,
        "false_negatives": 206,
        "true_negatives": 88095,
    },
    "limitations": [
        "PaySim is synthetic data.",
        "Raw model scores are not calibrated probabilities.",
        "Balance features may exploit simulator-specific patterns.",
    ],
}

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Model saved:", model_path)
print("Metadata saved:", metadata_path)
print("Model size:", round(model_path.stat().st_size / 1024, 2), "KB")
print("Saved threshold:", model_bundle["threshold"])

# Verify that the saved artifact can be loaded.
loaded_bundle = joblib.load(model_path)

assert loaded_bundle["features"] == list(selected_features)
assert loaded_bundle["threshold"] == float(selected_threshold)

print("Artifact reload verification: PASSED")


Model saved: D:\Prem\Projects\fraudshield\artifacts\fraud_model.joblib
Metadata saved: D:\Prem\Projects\fraudshield\artifacts\model_metadata.json
Model size: 364.53 KB
Saved threshold: 0.8053056330804439
Artifact reload verification: PASSED


In [7]:
import gc
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# --------------------------------------------------
# 1. Prepare the COMPLETE training partition
# --------------------------------------------------

X_all_train = X_train_full.loc[:, selected_features].astype(np.float32)

X_val_selected = X_val.loc[:, selected_features].astype(np.float32)
X_time_val_selected = X_time_val.loc[:, selected_features].astype(np.float32)
X_test_selected = X_final_test.loc[:, selected_features].astype(np.float32)

y_all_train = y_train_full.astype("uint8")

assert len(X_all_train) == len(y_all_train)

n_fraud = int((y_all_train == 1).sum())
n_legitimate = int((y_all_train == 0).sum())

print("FULL TRAINING PARTITION")
print("Total rows:", f"{len(X_all_train):,}")
print("Fraud cases:", f"{n_fraud:,}")
print("Legitimate cases:", f"{n_legitimate:,}")
print("Actual fraud rate:", f"{y_all_train.mean() * 100:.4f}%")

# Release source dataframes and the old sampled training subset.
# The split-specific feature frames and labels remain available.
del raw, features, target
del X_train_full, X_val, X_time_val, X_final_test
del X_train, y_train
gc.collect()

# --------------------------------------------------
# 2. Use sample weights instead of removing rows
# --------------------------------------------------

# Target an effective training ratio of approximately
# 1 fraud : 100 legitimate transactions.
negative_weight = (100.0 * n_fraud) / n_legitimate

sample_weights = np.where(
    y_all_train.to_numpy() == 1,
    1.0,
    negative_weight
).astype(np.float32)

print("\nLegitimate-row weight:", round(negative_weight, 6))
print("All training rows retained:", len(X_all_train))

# --------------------------------------------------
# 3. Train on ALL training rows
# --------------------------------------------------

full_data_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=True,
    random_state=42,
)

full_data_model.fit(
    X_all_train,
    y_all_train,
    sample_weight=sample_weights,
)

print("\nTraining completed.")
print("Iterations used:", full_data_model.n_iter_)

# --------------------------------------------------
# 4. Select the threshold on internal validation only
# --------------------------------------------------

full_val_prob = full_data_model.predict_proba(
    X_val_selected
)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(
    y_val, full_val_prob
)

f1_values = (
    2 * precisions[:-1] * recalls[:-1]
    / (precisions[:-1] + recalls[:-1] + 1e-12)
)

best_idx = int(np.argmax(f1_values))
full_data_threshold = float(thresholds[best_idx])

print("\nFULL-DATA MODEL")
print("Validation AP:", round(
    average_precision_score(y_val, full_val_prob), 4
))
print("Validation ROC-AUC:", round(
    roc_auc_score(y_val, full_val_prob), 4
))
print("Selected threshold:", round(full_data_threshold, 6))

# --------------------------------------------------
# 5. Compare both models on the SAME validation sets
# --------------------------------------------------

def evaluate_candidate(name, model):
    val_prob = model.predict_proba(X_val_selected)[:, 1]

    p, r, t = precision_recall_curve(y_val, val_prob)
    f1 = 2 * p[:-1] * r[:-1] / (
        p[:-1] + r[:-1] + 1e-12
    )
    threshold = float(t[int(np.argmax(f1))])

    later_prob = model.predict_proba(X_time_val_selected)[:, 1]
    later_pred = (later_prob >= threshold).astype("uint8")

    return {
        "model": name,
        "threshold": threshold,
        "validation_AP": average_precision_score(y_val, val_prob),
        "temporal_AP": average_precision_score(y_time_val, later_prob),
        "temporal_ROC_AUC": roc_auc_score(y_time_val, later_prob),
        "temporal_precision": precision_score(
            y_time_val, later_pred, zero_division=0
        ),
        "temporal_recall": recall_score(
            y_time_val, later_pred, zero_division=0
        ),
        "temporal_F1": f1_score(
            y_time_val, later_pred, zero_division=0
        ),
    }

comparison = pd.DataFrame([
    evaluate_candidate("Undersampled model", final_model),
    evaluate_candidate("Full-data weighted model", full_data_model),
])

display(comparison.round(4))

FULL TRAINING PARTITION
Total rows: 4,865,605
Fraud cases: 4,625
Legitimate cases: 4,860,980
Actual fraud rate: 0.0951%

Legitimate-row weight: 0.095145
All training rows retained: 4865605

Training completed.
Iterations used: 100

FULL-DATA MODEL
Validation AP: 0.8741
Validation ROC-AUC: 0.9991
Selected threshold: 0.780072


,model,threshold,validation_AP,temporal_AP,temporal_ROC_AUC,temporal_precision,temporal_recall,temporal_F1
0,Undersampled model,0.8053,0.8604,0.9148,0.9988,0.8425,0.8568,0.8496
1,Full-data weighted model,0.7801,0.8741,0.9258,0.9991,0.8567,0.8559,0.8563


Full Data Training

In [8]:

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Evaluate the full-data weighted model on the held-out test period.
# Do not change the threshold based on these results.
full_test_prob = full_data_model.predict_proba(
    X_test_selected[selected_features]
)[:, 1]

full_test_pred = (
    full_test_prob >= full_data_threshold
).astype("uint8")

print("FULL-DATA MODEL — FINAL TEST")
print("Threshold:", round(full_data_threshold, 6))
print("ROC-AUC:", round(
    roc_auc_score(y_final_test, full_test_prob), 4
))
print("Average precision:", round(
    average_precision_score(y_final_test, full_test_prob), 4
))
print("Fraud precision:", round(
    precision_score(y_final_test, full_test_pred, zero_division=0), 4
))
print("Fraud recall:", round(
    recall_score(y_final_test, full_test_pred, zero_division=0), 4
))
print("Fraud F1:", round(
    f1_score(y_final_test, full_test_pred, zero_division=0), 4
))

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_final_test, full_test_pred))

print("\nClassification report:")
print(classification_report(
    y_final_test,
    full_test_pred,
    target_names=["Legitimate", "Fraud"],
    zero_division=0,
))


FULL-DATA MODEL — FINAL TEST
Threshold: 0.780072
ROC-AUC: 0.9988
Average precision: 0.9447
Fraud precision: 0.9159
Fraud recall: 0.8355
Fraud F1: 0.8739

Confusion matrix [[TN, FP], [FN, TP]]:
[[88118    96]
 [  206  1046]]

Classification report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     88214
       Fraud       0.92      0.84      0.87      1252

    accuracy                           1.00     89466
   macro avg       0.96      0.92      0.94     89466
weighted avg       1.00      1.00      1.00     89466



In [9]:

from pathlib import Path
import json
import shutil
import joblib

# Reuse project_root if it was already defined.
if "project_root" not in globals():
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "raw").exists():
        project_root = cwd
    elif (cwd.parent / "data" / "raw").exists():
        project_root = cwd.parent
    else:
        raise FileNotFoundError("Could not locate the project root.")

artifacts_dir = project_root / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = artifacts_dir / "fraud_model.joblib"
metadata_path = artifacts_dir / "model_metadata.json"

# Preserve the earlier undersampled model and its metadata.
backup_model = artifacts_dir / "fraud_model_undersampled_v1.joblib"
backup_metadata = artifacts_dir / "model_metadata_undersampled_v1.json"

if model_path.exists() and not backup_model.exists():
    shutil.copy2(model_path, backup_model)
    print("Preserved previous model:", backup_model.name)

if metadata_path.exists() and not backup_metadata.exists():
    shutil.copy2(metadata_path, backup_metadata)
    print("Preserved previous metadata:", backup_metadata.name)

# Package the full-training-partition model.
model_bundle = {
    "model": full_data_model,
    "features": list(selected_features),
    "threshold": float(full_data_threshold),
    "model_name": "HistGradientBoostingClassifier",
    "training_strategy": "Full training partition with sample weights",
    "feature_version": "v2_full_training_weighted",
}

joblib.dump(model_bundle, model_path)

metadata = {
    "project": "FraudShield",
    "model_name": model_bundle["model_name"],
    "feature_version": model_bundle["feature_version"],
    "features": list(selected_features),
    "threshold": float(full_data_threshold),
    "threshold_selection": "Maximum F1 on internal validation",
    "training_rows": 4865605,
    "training_fraud_cases": 4625,
    "training_legitimate_cases": 4860980,
    "class_weighting": {
        "fraud": 1.0,
        "legitimate": 0.095145,
    },
    "final_temporal_evaluation": {
        "roc_auc": 0.9988,
        "average_precision": 0.9447,
        "fraud_precision": 0.9159,
        "fraud_recall": 0.8355,
        "fraud_f1": 0.8739,
        "true_negatives": 88118,
        "false_positives": 96,
        "false_negatives": 206,
        "true_positives": 1046,
    },
    "limitations": [
        "Training used the full training partition, not all dataset rows.",
        "PaySim is synthetic data.",
        "The model's raw scores are not calibrated fraud probabilities.",
        "Balance features may exploit simulator-specific patterns.",
        "The temporal test was used to compare candidate models."
    ],
}

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

# Verify the saved artifact loads and retains its configuration.
loaded = joblib.load(model_path)

assert loaded["features"] == list(selected_features)
assert loaded["threshold"] == float(full_data_threshold)
assert loaded["model"].n_features_in_ == len(selected_features)

print("\nPRIMARY MODEL SAVED")
print("Model:", model_path)
print("Metadata:", metadata_path)
print("Training rows:", f"{metadata['training_rows']:,}")
print("Features:", len(loaded["features"]))
print("Decision threshold:", loaded["threshold"])
print("Reload verification: PASSED")


Preserved previous model: fraud_model_undersampled_v1.joblib
Preserved previous metadata: model_metadata_undersampled_v1.json

PRIMARY MODEL SAVED
Model: D:\Prem\Projects\fraudshield\artifacts\fraud_model.joblib
Metadata: D:\Prem\Projects\fraudshield\artifacts\model_metadata.json
Training rows: 4,865,605
Features: 13
Decision threshold: 0.7800721293318664
Reload verification: PASSED


In [10]:

from pathlib import Path
import joblib
import sklearn
import numpy as np
import pandas as pd

# The notebook runs from the notebooks/ directory.
cwd = Path.cwd().resolve()

if (cwd / "artifacts" / "fraud_model.joblib").exists():
    project_root = cwd
elif (cwd.parent / "artifacts" / "fraud_model.joblib").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find artifacts/fraud_model.joblib"
    )

artifact_path = project_root / "artifacts" / "fraud_model.joblib"
bundle = joblib.load(artifact_path)

assert "model" in bundle
assert "features" in bundle
assert "threshold" in bundle

print("ARTIFACT:", artifact_path)
print("\nPACKAGE VERSIONS")
print("Python:", __import__("sys").version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Joblib:", joblib.__version__)

print("\nMODEL CONTRACT")
print("Model:", bundle["model_name"])
print("Feature version:", bundle["feature_version"])
print("Feature count:", len(bundle["features"]))
print("Feature order:", bundle["features"])
print("Threshold:", bundle["threshold"])
print("Model input feature count:",
      bundle["model"].n_features_in_)

assert len(bundle["features"]) == bundle["model"].n_features_in_

print("\nPREFLIGHT: PASSED")


ARTIFACT: D:\Prem\Projects\fraudshield\artifacts\fraud_model.joblib

PACKAGE VERSIONS
Python: 3.14.7
scikit-learn: 1.9.1
NumPy: 2.5.3
Pandas: 3.0.6
Joblib: 1.6.0

MODEL CONTRACT
Model: HistGradientBoostingClassifier
Feature version: v2_full_training_weighted
Feature count: 13
Feature order: ['amount', 'amount_log', 'hour', 'day', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'origin_balance_log', 'destination_balance_log', 'origin_balance_zero', 'destination_balance_zero']
Threshold: 0.7800721293318664
Model input feature count: 13

PREFLIGHT: PASSED
